In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr import Unit, Configuration, Variable

from utils.setup_utils import DEV_LOCATION_ID_LIST

### Setup

In [ ]:
spark = create_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# configure download
ev.download.configure(
    api_key='your_key_here'
)

### Add and populate domain tables

#### Pull domain tables from teehr warehouse

In [ ]:
print("Loading domain tables.")

print("Loading configurations.")
ev.download.configurations(
    load=True
)

print("Loading units.")
ev.download.units(
    load=True
)

print("Loading variables.")
ev.download.variables(
    load=True
)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading location attributes.")
ev.download.attributes(
    load=True
)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading domain variables, locations, crosswalks, and attributes complete!")

#### Add general domain entries to support FVED data ingest

In [ ]:
# add unit for volume
unit = Unit(
    name='m^3',
    long_name='Cubic Meter'
)
ev.units.add(unit)

#### Add domain entries to support cbrfc_unregulated_esp_forecast configuration data

In [ ]:
# add configurations for cbrfc wsv's (use existing for usgs/nwm)
configuration = Configuration(
    name="cbrfc_unregulated_esp_forecast",
    timeseries_type="secondary",
    description="CBRFC unregulated ESP water supply volume forecasts",
)
ev.configurations.add(configuration)

# add variable for monthly wsv's
variable = Variable(
    name="volume_monthly_sum",
    long_name="Monthly summed water supply volume"
)
ev.variables.add(variable)

#### Add domain entries to support cbrfc_unregulated_official_forecast configuration data (includes observed)

In [ ]:
# add new primary_locations to the locations table using existing usgs entries for now
existing_locs_gdf = ev.locations.to_geopandas()
new_locs = [
    {
        'id':'cbrfc-dlac2',
        'name':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09149500', 'name'].values[0], 
        'geometry':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09149500', 'geometry'].values[0],
    },
    {
        'id':'cbrfc-drgc2',
        'name':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09361500', 'name'].values[0], 
        'geometry':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09361500', 'geometry'].values[0],
    },
    {
        'id':'cbrfc-ydlc2',
        'name':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09260050', 'name'].values[0], 
        'geometry':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09260050', 'geometry'].values[0],
    },
]
new_locs_gdf = gpd.GeoDataFrame(new_locs, crs=existing_locs_gdf.crs)
ev.locations.load_dataframe(new_locs_gdf)


# add configurations for cbrfc_unregulated_official_forecast and observed
configuration = Configuration(
    name="cbrfc_unregulated_official_forecast",
    timeseries_type="secondary",
    description="CBRFC unregulated official water supply volume forecasts",
)
ev.configurations.add(configuration)
configuration = Configuration(
    name="cbrfc_unregulated_observed",
    timeseries_type="primary",
    description="CBRFC unregulated water-supply volume observations",
)
ev.configurations.add(configuration)


# add variables for seasonal wsv data to the variables table
# these were the three unique cases for CBRFC
variable_mapping = {
	'volume_amjj_sum':'Seasonal summed water supply volume (Apr-Jul)',
    'volume_jfmamj_sum':'Seasonal summed water supply volume (Jan-Jun)',
    'volume_jfmam_sum':'Seasonal summed water supply volume (Jan-May)',
}
for key in variable_mapping.keys():
    variable = Variable(
        name=key,
        long_name=variable_mapping[key],
    )
    ev.variables.add(variable)


# add crosswalk for cbrfc seasonal wsv data to the crosswalk table
cbrfc_to_cbrfc_dict = {
    'cbrfc-dlac2': 'cbrfc-dlac2', # DLAC2 in CBRFC 
    'cbrfc-drgc2': 'cbrfc-drgc2', # DRGC2 in CBRFC
    'cbrfc-ydlc2': 'cbrfc-ydlc2', # YDLC2 in CBRFC
}
crosswalk_dict = {
    'primary_location_id': cbrfc_to_cbrfc_dict.keys(),
    'secondary_location_id': cbrfc_to_cbrfc_dict.values()
}
temp = pd.DataFrame(crosswalk_dict)
ev.location_crosswalks.load_dataframe(temp)

#### Add domain entries to support nrcs_adjusted_forecast configurations data (includes observed)

In [ ]:
# add new primary_locations to the locations table using existing usgs entries for now
existing_locs_gdf = ev.locations.to_geodataframe()
new_locs = [
    {
        'id': 'nrcs-09361500:CO:USGS',
        'name': existing_locs_gdf.loc[existing_locs_gdf['id'] == 'usgs-09361500', 'name'].values[0],
        'geometry': existing_locs_gdf.loc[existing_locs_gdf['id'] == 'usgs-09361500', 'geometry'].values[0],
    },
    {
        'id': 'nrcs-09109209:CO:USGS',
        'name': existing_locs_gdf.loc[existing_locs_gdf['id'] == 'usgs-09109209', 'name'].values[0],
        'geometry': existing_locs_gdf.loc[existing_locs_gdf['id'] == 'usgs-09109209', 'geometry'].values[0],
    },
    {
        'id': 'nrcs-10128500:UT:USGS',
        'name': existing_locs_gdf.loc[existing_locs_gdf['id'] == 'usgs-10128500', 'name'].values[0],
        'geometry': existing_locs_gdf.loc[existing_locs_gdf['id'] == 'usgs-10128500', 'geometry'].values[0],
    }
]
new_locs_gdf = gpd.GeoDataFrame(new_locs, crs=existing_locs_gdf.crs)
ev.locations.add_geodataframe(new_locs_gdf)

# add configuration for nrcs_adjusted_forecast and observed
configuration = Configuration(
    name="nrcs_adjusted_forecast",
    timeseries_type="secondary",
    description="NRCS adjusted water supply volume forecasts",
)
ev.configurations.add(configuration)
configuration = Configuration(
    name="nrcs_adjusted_observed",
    timeseries_type="primary",
    description="NRCS adjusted water supply volume observations",
)
ev.configurations.add(configuration)

# add variables for nrcs_adjusted_forecast


# add crosswalk for nrcs test locs
nrcs_to_nrcs_dict = {
    'nrcs-09361500:CO:USGS': 'nrcs-09361500:CO:USGS', # DRGC2 in CBRFC
    'nrcs-09109209:CO:USGS': 'nrcs-09109209:CO:USGS', # TPIC2 in CBRFC
    'nrcs-10128500:UT:USGS': 'nrcs-10128500:UT:USGS', # OAWU1 in CBRFC
}
crosswalk_dict = {
    'primary_location_id': nrcs_to_nrcs_dict.keys(),
    'secondary_location_id': nrcs_to_nrcs_dict.values()
}
temp = pd.DataFrame(crosswalk_dict)
ev.location_crosswalks.load_dataframe(temp)

### Kill spark

In [ ]:
spark.stop()